In [ ]:
import pandas as pd
import os
import json
curr_wd = os.path.abspath(os.getcwd())
print(curr_wd)
from importlib import reload


In [ ]:
###### LOADING THE TWO DATASETS I NEED

In [ ]:
fpath = curr_wd + "/data/input/RG_motif_info_df.parquet"
motif_info_set_df = pd.read_parquet(fpath)
motif_info_set_df

In [ ]:
fpath = curr_wd + "/data/input/all_IDR_human.parquet"
IDR_info_df = pd.read_parquet(fpath)
IDR_info_df.rename(columns={"protein_name": "UniqueID"}, inplace=True)
IDR_info_df

In [ ]:
# from window_extraction import prepare_window_dataframe
import src.window_extraction as window_extraction
# Reload the module after making changes
reload(window_extraction)

# from fasta_writer import build_fasta_records, write_fasta_chunked

df_windows_neg = window_extraction.prepare_window_dataframe(
    df_motif=motif_info_set_df[motif_info_set_df["Groups_num"] == "7"],
    df_idr=IDR_info_df,
    flank=5,
    mode="adaptive",
    min_idr_fraction=0.9
)
print(df_windows_neg.head())
print(df_windows_neg.shape)

df_windows_pos = window_extraction.prepare_window_dataframe(
    df_motif=motif_info_set_df[motif_info_set_df["Groups_num"] == "5"],
    df_idr=IDR_info_df,
    flank=5,
    mode="adaptive",
    min_idr_fraction=0.9
)

print(df_windows_pos.head())
print(df_windows_pos.shape)

In [ ]:
###### SAVE THE METADATA FOR LATER

path = "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed"
filename = "pos_RG_regions_win5_metadata.pkl"

df_windows_pos.to_pickle(os.path.join(path, filename))

################################################

path = "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed"
filename = "neg_RG_regions_win5_metadata.pkl"

df_windows_neg.to_pickle(os.path.join(path, filename))


In [ ]:
#### GENERATE JSON FILES FOR GNOMAD

result_neg = df_windows_neg[["UniqueID", "win_start", "win_end"]].rename(
    columns={"win_start": "start", "win_end": "end"}
).to_dict(orient="records")

path = "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed"
filename = "neg_RG_regions_win5.json"


with open(f"{path}/{filename}", "w") as f:
    json.dump(result_neg, f, indent=2)

#######################################

result_pos = df_windows_pos[["UniqueID", "win_start", "win_end"]].rename(
    columns={"win_start": "start", "win_end": "end"}
).to_dict(orient="records")

path = "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed"
filename = "pos_RG_regions_win5.json"

with open(f"{path}/{filename}", "w") as f:
    json.dump(result_pos, f, indent=2)

In [ ]:
##### GENERATE FASTA FILES FOR BLASTP

import src.fasta_writer as fasta_writer
reload(fasta_writer)
# --- STEP 2: write FASTA ---
records_neg = fasta_writer.build_fasta_records(df_windows_neg)

path = "/mnt/d/phd/scripts/16_ev_signature_predictor/data/blastp"
filename = "neg_RG_regions_win5.fasta"

written_files = fasta_writer.write_fasta_chunked(
    records_neg,
    f"{path}/{filename}"
)

# ________________________

records_pos = fasta_writer.build_fasta_records(df_windows_pos)

path = "/mnt/d/phd/scripts/16_ev_signature_predictor/data/blastp"
filename = "pos_RG_regions_win5.fasta"

written_files = fasta_writer.write_fasta_chunked(
    records_pos,
    f"{path}/{filename}"
)